# 🍽️ 06 · Serve the model: API + Docker
### Servir el modelo: API + Docker

| | |
|---|---|
| 🍳 **Analogy / Analogía** | *The waiter (API) takes orders; the food-truck (Docker) can park anywhere* · *El mesero (API) toma pedidos; el food-truck (Docker) se estaciona en cualquier lugar* |
| 🎯 **Goal / Objetivo** | Let any program ask "UP or DOWN?" over HTTP · Que cualquier programa pregunte "¿SUBE o BAJA?" por HTTP |
| 🏭 **Becomes / Se convierte en** | `src/api/main.py`, `Dockerfile`, `docker-compose.yml` |

```
  📱 client / trading bot                🐳 Docker container
  ─────────────────────                  ┌────────────────────────────────────┐
  POST /predict  {last 300 candles} ───▶ │ FastAPI                            │
                                         │  1. make_features(with_target=False)│
  {"direction":"UP","prob_up":0.53} ◀─── │  2. champion.predict_proba()       │
                                         └────────────────────────────────────┘
```

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import json

import pandas as pd
from fastapi.testclient import TestClient

from src.api.main import app

C:\Users\Asus\Desktop\Claude Projects\MLOps Trading\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


**EN:** `TestClient` starts the API *inside this notebook* (no server needed) — the same thing the tests in CI do.
**ES:** `TestClient` levanta la API *dentro del notebook* (sin servidor) — lo mismo que hacen los tests en el CI.

In [2]:
client = TestClient(app)
client.__enter__()  # runs the startup: loads the champion model once

print("GET /health →", client.get("/health").json())

22:26:03 | INFO    | httpx | HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


GET /health → {'status': 'ok', 'model_loaded': True}


In [3]:
print(json.dumps(client.get("/model-info").json(), indent=2)[:800])

22:26:03 | INFO    | httpx | HTTP Request: GET http://testserver/model-info "HTTP/1.1 200 OK"


{
  "model": "xgboost",
  "version": 1,
  "registered_model": "gold-direction-h1",
  "trained_test_period": [
    "2025-09-24 06:00:00",
    "2026-09-29 04:00:00"
  ],
  "metrics": {
    "cv_roc_auc_mean": 0.5092,
    "test_accuracy": 0.5148,
    "test_precision": 0.5184,
    "test_recall": 0.6652,
    "test_f1": 0.5827,
    "test_roc_auc": 0.514,
    "test_strategy_return": 0.0527,
    "test_buyhold_return": 0.0982
  },
  "features": [
    "ret_1h",
    "ret_2h",
    "ret_3h",
    "ret_5h",
    "ret_10h",
    "ret_24h",
    "dist_sma_10",
    "dist_sma_50",
    "dist_sma_200",
    "rsi",
    "macd_hist",
    "volatility",
    "range_pct",
    "body_pct",
    "volume_z",
    "hour_sin",
    "hour_cos",
    "dayofweek"
  ]
}


## 🔮 Ask for a prediction / Pedir una predicción

In [4]:
candles = pd.read_csv("data/sample/xauusd_h1_sample.csv").tail(300)
payload = {"candles": candles[["time", "open", "high", "low", "close", "volume"]].to_dict(orient="records")}
r = client.post("/predict", json=payload)
print(r.status_code, r.json())

22:26:03 | INFO    | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


200 {'direction': 'DOWN', 'prob_up': 0.452, 'based_on_candle': '2026-09-29T05:00:00', 'model': 'xgboost', 'model_version': 1}


## 🛡️ The API protects itself / La API se protege sola
**EN:** Pydantic validates the input. Too few candles or a negative price → clear error (422), not a crash.
**ES:** Pydantic valida la entrada. Pocas velas o un precio negativo → error claro (422), no una caída.

In [5]:
bad = {"candles": payload["candles"][:10]}
r = client.post("/predict", json=bad)
print(r.status_code, r.json()["detail"][0]["msg"])

22:26:03 | INFO    | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


422 List should have at least 250 items after validation, not 10


In [6]:
client.__exit__(None, None, None)

## 🐳 Now in a container / Ahora en un contenedor

```powershell
docker compose up --build          # builds the image and starts the API
# open http://localhost:8000/docs  → interactive Swagger page ("Try it out")
python scripts/live_demo.py        # real MT5 candles → API → prediction for the next hour
```

**EN — Why Docker?** "It works on my machine" disappears: the image contains Python, the libraries (exact
versions), the code and the model. The same image runs on your laptop, in CI and on a cloud server.

**ES — ¿Por qué Docker?** Se acaba el "en mi máquina funciona": la imagen contiene Python, las librerías (versiones
exactas), el código y el modelo. La misma imagen corre en tu laptop, en el CI y en un servidor en la nube.

---
## 🔁 From notebook to production / Del notebook a producción

| Notebook | Production |
|---|---|
| `TestClient` in a cell | `tests/test_api.py` — runs on every push |
| Model loaded by hand | Loaded **once** at startup (`lifespan`) from `MODEL_DIR` |
| Python on your PC | `Dockerfile` (slim image, non-root user, healthcheck) |
| — | **CD** pipeline builds & publishes the image to `ghcr.io` automatically |